# 05 — LSTM Model (Khushi's Part)
Roadmap steps covered here: **12 (scale + sequence data), 13 (build + tune LSTM on all folds), 14 (backtest LSTM)**.

Input: `data/processed/nifty50_labeled.csv` (do not modify).
Fold structure: `src/validation.py` (do not redefine).
Backtest engine: `src/backtest.py` (do not redefine).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.validation import get_walk_forward_folds, get_test_split
# from src.backtest import run_backtest, run_walk_forward_backtest, plot_all_folds  # not available yet — Sneha's part
from src.models.lstm_model import (
    FEATURE_COLS, DEFAULT_LOOKBACK,
    train_and_evaluate_fold, tune_hyperparameters, run_final_test,
    save_predictions,
)

pd.set_option('display.max_columns', None)

## Load data
This is the single input file everyone shares — do not modify it, and do not touch the 2024-2026 rows until the final test cell at the bottom.

In [ ]:
df = pd.read_csv("../data/processed/nifty50_labeled.csv", index_col=0, parse_dates=True)
df = df.sort_index()
print(df.shape)
df.head()

## Walk-forward folds
Using the shared `src/validation.py` — 4 expanding folds, 2020-2023 as validation years.

In [ ]:
folds = get_walk_forward_folds(df)

## Step 13c: Hyperparameter tuning
Small grid over `lookback`, `lstm_units`, `dropout`, `learning_rate`.
Each config is scored by **average macro-F1 across all 4 folds** (not accuracy — a model that
always predicts NEUTRAL can hit 50-60% accuracy and still be useless, per the roadmap's
class-imbalance warning). The 2024-2026 test set is never touched in this step.

Start with a small `epochs` budget while iterating, then raise it once you've picked a
short-list of configs to confirm.


In [ ]:
tuning_results = tune_hyperparameters(folds, feature_cols=FEATURE_COLS, epochs=20, verbose=0)
tuning_results


In [ ]:
best_row = tuning_results.iloc[0]
best_params = {
    "lookback": int(best_row["lookback"]),
    "lstm_units": best_row["lstm_units"],
    "dropout": float(best_row["dropout"]),
    "learning_rate": float(best_row["learning_rate"]),
}
print("Best config (freeze this):", best_params)

## Step 13b: Full run of the frozen config on all 4 folds
Re-run (with more epochs) so we keep the trained models, predictions, and confusion matrices for the report.

In [ ]:
best_row = tuning_results.iloc[0]
best_params = {
    "lookback": int(best_row["lookback"]),
    "lstm_units": best_row["lstm_units"],
    "dropout": float(best_row["dropout"]),
    "learning_rate": float(best_row["learning_rate"]),
}
print("Best config (freeze this):", best_params)

## Save per-fold predictions
Matches `src/backtest.py`'s expected CSV contract: `Date` + `Predicted_Target` in {0,1,2}.

In [ ]:
os.makedirs("../data/predictions", exist_ok=True)
for res in fold_results:
    fname = f"../data/predictions/lstm_{res['fold'].lower().replace(' ', '_')}.csv"
    save_predictions(res["predictions"], fname)
    print("saved", fname)


## Step 14: Backtest the LSTM across all folds
Uses `src/backtest.py` directly — no re-implementation.

In [ ]:
predictions_by_fold = {res["fold"]: res["predictions"] for res in fold_results}
combined = run_walk_forward_backtest(df, predictions_by_fold)
combined.summary()

In [ ]:
plot_all_folds(combined)


## Results table (classification + trading metrics per fold)
Saved to `results/lstm_fold_metrics.csv` for Sneha's final comparison table (step 16).


In [ ]:
rows = []
for res, bt in zip(fold_results, combined.fold_results):
    rows.append({
        "fold": res["fold"],
        "accuracy": res["accuracy"],
        "macro_f1": res["macro_f1"],
        "sharpe": bt.sharpe,
        "cagr": bt.cagr,
        "max_drawdown": bt.max_drawdown,
        "n_trades": bt.n_trades,
    })
lstm_results = pd.DataFrame(rows)
os.makedirs("../results", exist_ok=True)
lstm_results.to_csv("../results/lstm_fold_metrics.csv", index=False)
lstm_results

## Final test (touch ONLY after hyperparameters are frozen above)
Retrain on the full 2015-2023 window with the frozen `best_params`, evaluate once on 2024-2026.
Per the roadmap's non-negotiable rules: **the test set is touched exactly once** — do not re-run
this cell with different hyperparameters after seeing the result.


In [ ]:
train_df, test_df = get_test_split(df)
final_result = run_final_test(train_df, test_df, best_params, FEATURE_COLS, epochs=80)
print(f"FINAL TEST -> accuracy={final_result['accuracy']:.3f}  macro_f1={final_result['macro_f1']:.3f}")
print(final_result["confusion_matrix"])

save_predictions(final_result["predictions"], "../data/predictions/lstm_final_test.csv")
final_backtest = run_backtest(df, final_result["predictions"], label="LSTM Final Test")
final_backtest.summary()
final_backtest.plot_equity_curve()


In [ ]:
final_result["model"].save("../saved_models/lstm_best.keras")
print("saved ../saved_models/lstm_best.keras")
